# Ejercicio 11: Reflexión final

Respuestas breves apoyadas en ecuaciones y en resultados numéricos del trabajo. La celda de código recalcula las cantidades que se citan: rango, espacio nulo, errores con ruido, regularización y métodos de optimización.

In [1]:
import numpy as np

# la celda de siempre: fantoma y matrices de los ej. anteriores, las redefino pq cada notebook tiene que andar solo
X_fantoma = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
    [0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
    [0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)
n = 8
x_fantoma = X_fantoma.reshape(-1)


def mascara_rayo(n, direccion, s):
    """Matriz n x n con 1 donde alpha*i + beta*j == s (centros de pixel)."""
    alpha, beta = direccion
    i, j = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    return (alpha * i + beta * j == s).astype(float)


def matriz_proyeccion(n, direcciones):
    """Devuelve (A, rayos): A en R^{m x n^2} y la lista [((alpha, beta), s), ...]."""
    i, j = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    filas, rayos = [], []
    for (alpha, beta) in direcciones:
        valores_s = np.unique(alpha * i + beta * j)  # ordenados de menor a mayor
        for s in valores_s:
            filas.append(mascara_rayo(n, (alpha, beta), s).reshape(-1))
            rayos.append(((alpha, beta), int(s)))
    return np.array(filas), rayos


D2 = [(1, 0), (0, 1)]
D4 = D2 + [(1, 1), (1, -1)]
D7 = D4 + [(2, 1), (1, 2), (3, 1)]
A2, rayos2 = matriz_proyeccion(n, D2)
A4, rayos4 = matriz_proyeccion(n, D4)
A7, rayos7 = matriz_proyeccion(n, D7)

# Mediciones ideales y ruido (misma semilla y mismo z en todos los ejercicios)
b7 = A7 @ x_fantoma
escala = np.sqrt(np.mean(b7 ** 2))
# misma semilla que en el ej. 6, si no los numeros no coinciden con los de alla
rng = np.random.default_rng(912)
z = rng.standard_normal(b7.shape)


def medicion_ruidosa(sigma):
    return b7 + sigma * escala * z

In [2]:
# ---- Resultados numéricos que se citan en las respuestas ----
nx = np.linalg.norm(x_fantoma)
print("== Rango y espacio nulo ==")
for nombre, A in [("A2", A2), ("A4", A4), ("A7", A7)]:
    rk = np.linalg.matrix_rank(A)
    print(f"{nombre}: {A.shape}, rango = {rk}, dim Nul = {64 - rk}")

print("\n== Reconstrucción ideal con lstsq (mín. norma) ==")
for nombre, A in [("A2", A2), ("A4", A4), ("A7", A7)]:
    xh = np.linalg.lstsq(A, A @ x_fantoma, rcond=None)[0]
    print(f"{nombre}: ||residuo|| = {np.linalg.norm(A @ x_fantoma - A @ xh):.1e}, error relativo = {np.linalg.norm(xh - x_fantoma)/nx:.3f}")

print("\n== Ruido con A7 (mínimos cuadrados) ==")
for sg in [0, 0.01, 0.03, 0.06, 0.10]:
    b_ = medicion_ruidosa(sg); xh = np.linalg.lstsq(A7, b_, rcond=None)[0]
    print(f"sigma={sg:<5}: ||ruido|| = {np.linalg.norm(b_-b7):.3f}, ||residuo|| = {np.linalg.norm(b_-A7@xh):.3f}, error rel. = {np.linalg.norm(xh-x_fantoma)/nx:.3f}")

print("\n== Regularización (gradiente descendiente, alpha = 1/L) ==")
# armo D rapido, es la misma del ej. 9
D = np.zeros((112, 64)); r = 0
for i in range(8):
    for j in range(7): D[r, i*8+j], D[r, i*8+j+1] = -1, 1; r += 1
for i in range(7):
    for j in range(8): D[r, i*8+j], D[r, i*8+j+8] = -1, 1; r += 1
AtA, DtD = A7.T @ A7, D.T @ D
def solve_reg(b, lam):
    x = np.zeros(64); a = 1 / np.linalg.eigvalsh(AtA + DtD)[-1]; Atb = A7.T @ b
    for t in range(20000):
        g = AtA @ x - Atb + lam * DtD @ x
        if np.linalg.norm(g) <= 1e-8 * np.linalg.norm(Atb): break
        x -= a * g
    return x, t
for sg in [0.0, 0.10, 0.20]:
    b_ = b7 + sg * escala * z
    print(f"sigma={sg:.2f}: " + "  ".join(f"lam={lam}: err={np.linalg.norm(solve_reg(b_, lam)[0]-x_fantoma)/nx:.3f}" for lam in [0, 0.1, 0.3, 1]))

print("\n== Métodos para el problema de mínimos cuadrados (A7, sigma = 0.06) ==")
b_ = medicion_ruidosa(0.06); H = AtA; mu = np.linalg.eigvalsh(H)
x_ls = np.linalg.lstsq(A7, b_, rcond=None)[0]
x_n = np.linalg.solve(H, A7.T @ b_)                      # Newton desde x0=0: un solo paso
# gradiente descendiente a mano, alpha = 1/mu_max
x = np.zeros(64)
for t in range(100000):
    g = H @ x - A7.T @ b_
    if np.linalg.norm(g) <= 1e-10 * np.linalg.norm(A7.T @ b_): break
    x -= g / mu[-1]
print(f"mu_max = {mu[-1]:.2f}, mu_min = {mu[0]:.3f}, cond(H) = {mu[-1]/mu[0]:.1f}")
print(f"lstsq: directo | GD: {t} iteraciones (alpha = 1/mu_max) | Newton: 1 iteración")
print(f"||x_GD - x_ls|| = {np.linalg.norm(x-x_ls):.1e},  ||x_Newton - x_ls|| = {np.linalg.norm(x_n-x_ls):.1e}")

== Rango y espacio nulo ==
A2: (16, 64), rango = 15, dim Nul = 49
A4: (46, 64), rango = 39, dim Nul = 25
A7: (119, 64), rango = 64, dim Nul = 0

== Reconstrucción ideal con lstsq (mín. norma) ==
A2: ||residuo|| = 4.0e-15, error relativo = 0.469
A4: ||residuo|| = 1.6e-14, error relativo = 0.136
A7: ||residuo|| = 1.3e-14, error relativo = 0.000

== Ruido con A7 (mínimos cuadrados) ==
sigma=0    : ||ruido|| = 0.000, ||residuo|| = 0.000, error rel. = 0.000
sigma=0.01 : ||ruido|| = 0.133, ||residuo|| = 0.089, error rel. = 0.021
sigma=0.03 : ||ruido|| = 0.398, ||residuo|| = 0.267, error rel. = 0.063
sigma=0.06 : ||ruido|| = 0.795, ||residuo|| = 0.534, error rel. = 0.126
sigma=0.1  : ||ruido|| = 1.326, ||residuo|| = 0.889, error rel. = 0.210

== Regularización (gradiente descendiente, alpha = 1/L) ==
sigma=0.00: lam=0: err=0.000  lam=0.1: err=0.042  lam=0.3: err=0.097  lam=1: err=0.200
sigma=0.10: lam=0: err=0.210  lam=0.1: err=0.171  lam=0.3: err=0.163  lam=1: err=0.224
sigma=0.20: lam=0: er

## Inciso a) Por qué es un problema inverso

En el problema directo se conoce la imagen x (la distribución de atenuación) y el modelo A, y se calculan las mediciones b = A x: es lo que hace el escáner físico. En la tomografía conocemos las mediciones b (el sinograma) y la geometría de adquisición, que está codificada en A, y se busca recuperar la imagen x del interior del objeto. Ir de los efectos observados a las x desconocidas es por definición, un problema inverso.

El modelo b = A x es el puente: dice qué mediciones produciría cada imagen candidata, y por lo tanto permite comparar candidatas con los datos. La solución puede no ser única (si el espacio nulo de A no es solo el cero), puede no existir (con ruido, b no está en el espacio columna de A) y puede ser inestable (pequeños cambios en b producen grandes cambios en x cuando A está mal condicionada).

## Inciso b) Cuándo las mediciones determinan una única imagen

Si A tiene m filas y n^2 columnas, el teorema de rango y nulidad dice que rango(A) + dim Nul(A) = n^2. La imagen es única si y solo si Nul(A) = {0}, es decir, si rango(A) = n^2 (rango columna completo). Si existe v distinto de cero con A v = 0, entonces A(x + gamma v) = A x para todo gamma: infinitas imágenes distintas producen exactamente las mismas mediciones, así que ninguna medición puede distinguirlas. Esas componentes son invisibles para el escáner.

Numéricamente (celda de arriba), con n^2 = 64: A2 tiene 16 filas, rango 15 y dim Nul 49, así que no es única. A4 tiene 46 filas, rango 39 y dim Nul 25, tampoco es única. A7 tiene 119 filas, rango 64 y dim Nul 0, y sí es única.

Tener más filas no basta: A4 tiene 46 filas pero rango 39, porque hay mediciones redundantes (combinaciones lineales de otras). Importa el rango, que depende de que las orientaciones aporten información nueva. Un residuo nulo tampoco garantiza recuperar la imagen: con A2 y A4, lstsq reproduce las mediciones exactamente (residuo del orden de 1e-14) pero con error relativo grande respecto del fantoma, porque devuelve la solución de norma mínima, que no tiene componente en Nul(A), mientras que el fantoma sí la tiene.

## Inciso c) Efecto del ruido y mínimos cuadrados

Las mediciones reales son b_sigma = A x + eta. En general b_sigma no está en el espacio columna de A: el ruido saca al vector de medidas del conjunto de mediciones posibles, y el sistema A x = b_sigma no tiene solución exacta, aunque A tenga rango columna completo, porque hay más ecuaciones que incógnitas.

Mínimos cuadrados resuelve el problema de minimizar ||A x - b_sigma||^2: busca la imagen cuyas mediciones simuladas están lo más cerca posible de las observadas, y la solución satisface las ecuaciones normales A^T A x = A^T b_sigma. Geométricamente, A x_hat es la proyección ortogonal de b_sigma sobre el espacio columna de A.

El residuo r = b_sigma - A x_hat es la parte de b_sigma ortogonal al espacio columna de A. Como b = A x_real está en ese espacio, resulta r = (I - P) eta, la componente del ruido que el modelo no puede explicar. Su norma es del orden de ||eta|| (con sigma = 0,06: ||eta|| = 0,795 y ||r|| = 0,534).

Alcanza la norma del residuo para evaluar la calidad? No, en dos sentidos opuestos. Un residuo pequeño no implica buena reconstrucción: con A2 o A4 el residuo es del orden de 1e-14 y el error es grande (por el espacio nulo); y si el sistema está mal condicionado, ajustar demasiado bien significa ajustar el ruido (el residuo del problema sin regularizar es el más chico, pero no el de menor error). Por otro lado, un residuo comparable al ruido es lo esperable de una buena solución. En el experimento con A7 el error relativo crece linealmente con sigma (0,021; 0,063; 0,126 y 0,210 para sigma = 0,01; 0,03; 0,06 y 0,10) y el residuo crece en la misma proporción, de modo que el residuo mide el nivel de ruido pero no dice cuánto del error está en la imagen.

## Inciso d) El supuesto del término de regularización

La norma al cuadrado de D x es la suma de los cuadrados de las diferencias entre píxeles vecinos. Introduce el supuesto a priori de que la imagen es suave: los píxeles vecinos tienen atenuaciones parecidas y los saltos grandes son raros, mientras que el ruido produce diferencias grandes y erráticas entre píxeles contiguos. El funcional J busca una imagen que ajuste los datos pero sin variaciones innecesarias. La penalización es cuadrática, así que penaliza todos los saltos, incluidos los bordes reales.

Efecto de lambda en el Ejercicio 9, con A7, en errores relativos. Con sigma = 0: 0,000 para lambda = 0; 0,042 para 0,1; 0,097 para 0,3; 0,200 para 1. Con sigma = 0,10: 0,210; 0,171; 0,163 (el mejor); 0,224. Con sigma = 0,20: 0,420; 0,337; 0,281; 0,278 (el mejor).

Con lambda chico hay poca suavización: con ruido la imagen queda granular, porque la varianza del ruido pasa a la solución. Con lambda grande la solución se acerca a una imagen casi constante, los bordes se difuminan y los picos se aplanan; eso introduce sesgo (incluso sin ruido el error crece con lambda) y aumenta el residuo. El mejor lambda aumenta con el ruido (0, 0,3 y 1 dentro de la grilla): es el compromiso entre sesgo y varianza.

## Inciso e) lstsq, gradiente descendiente y Newton

Iteraciones: lstsq es un método directo (SVD); gradiente descendiente necesita unas 1200 iteraciones para tolerancia 1e-10 con alpha = 1/mu_max; Newton necesita una.

Parámetros: lstsq no tiene ninguno (opcionalmente rcond); gradiente descendiente necesita 0 < alpha < 2/mu_max, un criterio de parada y el punto inicial; Newton no tiene ninguno, solo resuelve H p = gradiente.

Costo por paso: lstsq hace una factorización completa, una sola vez; gradiente descendiente hace dos productos matriz-vector (barato, y escala a problemas ralos); Newton resuelve un sistema de n^2 x n^2.

Solución: los tres dan la solución de mínimos cuadrados (la de norma mínima si no hay unicidad), con error del orden de la tolerancia para gradiente descendiente y hasta redondeo para Newton. Para A7, que tiene rango columna completo, los tres coinciden: las diferencias son del orden de 1e-8 para gradiente descendiente y de 1e-15 para Newton.

Sobre Newton: como E es cuadrática, su modelo de segundo orden es exacto y H = A^T A es constante, así que el paso x1 = x0 - H^(-1) * gradiente(x0) cae directamente en el minimizador. En una función no cuadrática eso no ocurre, y además resolver con H de n^2 x n^2 es costoso para imágenes grandes, por eso en el Ejercicio 10 se usa gradiente descendiente con matrices dispersas. La cantidad de iteraciones de gradiente descendiente depende del número de condición mu_max/mu_min, que acá es de aproximadamente 71.

## Inciso f) Síntesis

El modelo directo es b = A x, y el problema inverso regularizado es encontrar x_hat que minimice 1/2 * ||A x - b||^2 + lambda/2 * ||D x||^2. Cada concepto del curso aporta algo distinto a este problema.

El modelo b = A x muestra cómo cada rayo mide una suma ponderada de píxeles; A (de m x n^2, rala) codifica la geometría.

El rango dice cuánta información independiente aportan las mediciones. Que rango(A) sea n^2 es necesario para una reconstrucción única, y las filas redundantes no suman rango.

El espacio nulo dice qué imágenes son invisibles para el escáner (A v = 0) y por lo tanto no pueden recuperarse solo con los datos; explica por qué un residuo nulo no basta (A2 y A4).

Las proyecciones ortogonales dan la geometría de la solución con ruido: A x_hat es la proyección de b sobre el espacio columna de A y el residuo es ortogonal a ese espacio (A^T r = 0).

Mínimos cuadrados es la forma de manejar datos inconsistentes por el ruido: la imagen cuyas mediciones simuladas se acercan más a las observadas, a través de las ecuaciones normales.

Las matrices simétricas y los valores propios: A^T A (la Hessiana) es simétrica semidefinida positiva; mu_max acota la tasa de aprendizaje (alpha < 2/mu_max) y el cociente mu_max/mu_min gobierna la velocidad y el mal condicionamiento (la amplificación del ruido).

Gradiente descendiente y Newton son dos formas de calcular la solución: iterativa y escalable (gradiente) o directa en un paso para cuadráticas (Newton).

La regularización permite incorporar conocimiento previo (suavidad) para estabilizar la solución frente al ruido y la falta de unicidad, a costa de un sesgo controlado por lambda: es el compromiso entre fidelidad, ruido y detalle.